# YouTube → Dailymotion (Kaggle)

Downloads one video, splits it (~1h 55m), uploads to Dailymotion API v2, then deletes local files.

**Dailymotion standard (free) caps:** 15 videos or 10 hours per 24 hours, and 2 hours / 4 GB per file. The worker stops before exceeding those, saves every uploaded URL plus the resume timestamp, and Admin **Resume** continues from that point after the daily window resets.

**One-time setup**

1. Settings → turn **Internet** on.
2. In [Dailymotion Studio](https://studio.dailymotion.com/) → Organization → API keys, create a **Private API key** (password login is no longer supported).
3. Add-ons → Secrets:
   - `API_BASE` — `https://your-backend.vercel.app/api`
   - `COLAB_JOB_SECRET`
   - `DAILYMOTION_CLIENT_ID` — private API key
   - `DAILYMOTION_CLIENT_SECRET` — private API secret
   - `DAILYMOTION_PROFILE_ID` — optional; taken from `/v2/me` if omitted
   - `YOUTUBE_COOKIES` — optional Netscape `cookies.txt` contents if you are not attaching a dataset
4. Attach a private dataset that contains `cookies.txt` (recommended). YouTube downloads usually 403 without cookies.
5. Save. `POST /api/colab-jobs` starts this notebook; you do not need to leave it open.


In [ ]:
#!/usr/bin/env python3
"""Kaggle worker: download YouTube → split → Dailymotion → cleanup.

Designed for Kaggle's ~20GB /kaggle/working disk. One video (and one split
part) is kept on disk at a time; files are deleted as soon as they are uploaded.
"""

from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

WORK_DIR = Path("/kaggle/working/ytdm")
COOKIE_FILENAMES = ("cookies.txt", "combined_cookies.txt", "youtube_cookies.txt")
COOKIE_CANDIDATES = [
    "/kaggle/input/youtube-cookies/cookies.txt",
    "/kaggle/input/youtube-cookies/combined_cookies.txt",
    "/kaggle/input/youtube-cookies/youtube_cookies.txt",
]
QUALITY_FORMATS = {
    "Best Available": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "4K": "bestvideo[height<=2160]+bestaudio/best[height<=2160]/best",
    "1080p": "bestvideo[height<=1080]+bestaudio/best[height<=1080]/best",
    "720p": "bestvideo[height<=720]+bestaudio/best[height<=720]/best",
    "480p": "bestvideo[height<=480]+bestaudio/best[height<=480]/best",
    "360p": "bestvideo[height<=360]+bestaudio/best[height<=360]/best",
}
# YouTube currently 403s android_sdkless googlevideo URLs. Deno is required
# for JS challenges. Combined `best` is a last-resort format if DASH 403s.
YOUTUBE_DOWNLOAD_ATTEMPTS = (
    {
        "label": "default clients (no android_sdkless)",
        "extractor_args": {"youtube": {"player_client": ["default", "-android_sdkless"]}},
        "format_key": "quality",
    },
    {
        "label": "tv + mweb clients",
        "extractor_args": {"youtube": {"player_client": ["tv", "mweb", "web"]}},
        "format_key": "quality",
    },
    {
        "label": "combined progressive stream",
        "extractor_args": {"youtube": {"player_client": ["default", "-android_sdkless"]}},
        "format_key": "combined",
    },
)
# Dailymotion Standard (free) account caps.
DM_MAX_FILE_BYTES = 4 * 1024 * 1024 * 1024
DM_MAX_FILE_SECONDS = 2 * 3600
DM_MAX_VIDEOS_PER_DAY = 15
DM_MAX_SECONDS_PER_DAY = 10 * 3600
DAILY_LIMIT_HINTS = ("limit", "quota", "too many", "daily", "rate limit")
PART_PATTERN = re.compile(r"^(.*) - Part (\d+)$")


class DailyLimitReached(Exception):
    pass


class JobCheckpoint:
    def __init__(self, urls: list[dict], next_part: int, resume_at_seconds: float) -> None:
        self.urls = list(urls)
        self.next_part = next_part
        self.resume_at_seconds = float(resume_at_seconds)

    @classmethod
    def from_job(cls, job: dict) -> "JobCheckpoint":
        return cls(
            urls=list(job.get("resultUrls") or []),
            next_part=int(job.get("startPartNumber") or 1),
            resume_at_seconds=float(job.get("resumeAtSeconds") or 0),
        )

    def payload(self) -> dict:
        return {
            "urls": self.urls,
            "nextStartPart": self.next_part,
            "resumeAtSeconds": int(self.resume_at_seconds),
        }


class PipelineStop(Exception):
    def __init__(self, message: str, checkpoint: JobCheckpoint, *, paused: bool) -> None:
        super().__init__(message)
        self.checkpoint = checkpoint
        self.paused = paused


class SessionQuota:
    def __init__(self) -> None:
        self.videos = 0
        self.seconds = 0.0

    @classmethod
    def from_job(cls, job: dict) -> "SessionQuota":
        quota = cls()
        completed = job.get("completedAt") or job.get("updatedAt")
        if not completed:
            return quota
        try:
            from datetime import datetime, timezone

            stamp = (
                datetime.fromisoformat(str(completed).replace("Z", "+00:00"))
                if not isinstance(completed, datetime)
                else completed
            )
            if stamp.tzinfo is None:
                stamp = stamp.replace(tzinfo=timezone.utc)
            age = datetime.now(timezone.utc) - stamp.astimezone(timezone.utc)
            if age.total_seconds() > 24 * 3600:
                return quota
        except Exception:
            return quota
        quota.videos = len(job.get("resultUrls") or [])
        quota.seconds = float(job.get("resumeAtSeconds") or 0)
        if quota.videos or quota.seconds:
            log(
                f"📊 Counting {quota.videos} saved upload(s) / {format_hms(quota.seconds)} "
                "toward today's Dailymotion cap (paused within 24h)"
            )
        return quota

    def would_exceed(self, part_seconds: float) -> str | None:
        if part_seconds > DM_MAX_FILE_SECONDS + 1:
            return (
                f"Part length {format_hms(part_seconds)} exceeds Dailymotion's "
                "2-hour per-file limit."
            )
        if self.videos >= DM_MAX_VIDEOS_PER_DAY:
            return (
                f"Dailymotion standard daily video cap ({DM_MAX_VIDEOS_PER_DAY} videos / 24h) "
                "would be exceeded."
            )
        if self.seconds + part_seconds > DM_MAX_SECONDS_PER_DAY + 1:
            return (
                f"Dailymotion standard daily duration cap ({DM_MAX_SECONDS_PER_DAY // 3600} hours / 24h) "
                "would be exceeded."
            )
        return None

    def record(self, part_seconds: float) -> None:
        self.videos += 1
        self.seconds += part_seconds


def format_hms(seconds: float) -> str:
    total = max(0, int(seconds))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h{minutes:02d}m{secs:02d}s"
    return f"{minutes}m{secs:02d}s"


def log(message: str) -> None:
    print(message, flush=True)


def disk_status(path: Path) -> None:
    usage = shutil.disk_usage(path)
    log(f"💾 Disk: {usage.used / 1e9:.1f} GB used, {usage.free / 1e9:.1f} GB free")


def remove_path(path: str | Path | None) -> None:
    if not path:
        return
    target = Path(path)
    try:
        if target.is_dir():
            shutil.rmtree(target, ignore_errors=True)
        elif target.exists():
            target.unlink()
    except OSError as err:
        log(f"⚠️ Could not delete {target}: {err}")


def get_secret(name: str, *aliases: str) -> str:
    for key in (name, *aliases):
        value = os.environ.get(key, "").strip()
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()
        for key in (name, *aliases):
            try:
                value = (client.get_secret(key) or "").strip()
            except Exception:
                value = ""
            if value:
                return value
    except Exception:
        pass
    return ""


def ensure_tools() -> None:
    log("📦 Installing yt-dlp, tqdm, requests-toolbelt...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "yt-dlp[default]",
            "tqdm",
            "requests-toolbelt",
        ],
        check=True,
    )
    deno_bin = str(Path.home() / ".deno" / "bin")
    if deno_bin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")
    if deno_path() is None:
        log("📦 Installing Deno (yt-dlp YouTube JS challenges)...")
        subprocess.run(
            ["bash", "-c", "curl -fsSL https://deno.land/install.sh | sh -s -- -y"],
            check=True,
        )
    try:
        import yt_dlp

        log(f"yt-dlp {yt_dlp.version.__version__}")
    except Exception:
        pass
    deno = deno_path()
    if deno:
        log(f"🦕 Deno JS runtime: {deno}")
    else:
        log("⚠️ Deno not on PATH — YouTube JS challenges will fail")


def api_headers(secret: str) -> dict[str, str]:
    return {"x-colab-secret": secret, "Content-Type": "application/json"}


def claim_job(api_base: str, secret: str) -> dict | None:
    import requests

    res = requests.get(
        f"{api_base.rstrip('/')}/colab-jobs/pending",
        headers=api_headers(secret),
        timeout=30,
    )
    res.raise_for_status()
    return res.json().get("job")


def complete_job(api_base: str, secret: str, job_id: int, urls: list[dict]) -> None:
    import requests

    res = requests.post(
        f"{api_base.rstrip('/')}/colab-jobs/{job_id}/complete",
        headers=api_headers(secret),
        json={"urls": urls},
        timeout=30,
    )
    res.raise_for_status()


def save_progress(api_base: str, secret: str, job_id: int, checkpoint: JobCheckpoint) -> None:
    import requests

    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/progress",
            headers=api_headers(secret),
            json=checkpoint.payload(),
            timeout=30,
        ).raise_for_status()
        log(
            f"💾 Checkpoint: {len(checkpoint.urls)} URL(s), next part {checkpoint.next_part} "
            f"at {format_hms(checkpoint.resume_at_seconds)}"
        )
    except Exception as err:
        log(f"⚠️ Could not save progress to backend: {err}")


def fail_job(
    api_base: str,
    secret: str,
    job_id: int,
    error_message: str,
    *,
    paused: bool = False,
    checkpoint: JobCheckpoint | None = None,
) -> None:
    import requests

    payload: dict = {"errorMessage": error_message[:4000], "paused": paused}
    if checkpoint is not None:
        payload.update(checkpoint.payload())
    try:
        requests.post(
            f"{api_base.rstrip('/')}/colab-jobs/{job_id}/fail",
            headers=api_headers(secret),
            json=payload,
            timeout=30,
        ).raise_for_status()
    except Exception as err:
        log(f"⚠️ Could not report failure to backend: {err}")


def deno_path() -> str | None:
    candidates = [
        shutil.which("deno"),
        str(Path.home() / ".deno" / "bin" / "deno"),
        "/root/.deno/bin/deno",
        "/usr/local/bin/deno",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).is_file() and os.access(candidate, os.X_OK):
            return candidate
    return None


def write_cookie_file(raw: str, dest: Path) -> str:
    dest.write_text(raw.replace("\\n", "\n"), encoding="utf-8")
    return str(dest)


def fetch_admin_cookies(api_base: str, secret: str) -> str | None:
    import requests

    try:
        res = requests.get(
            f"{api_base.rstrip('/')}/colab-jobs/cookies",
            headers=api_headers(secret),
            timeout=30,
        )
        if not res.ok:
            log(f"⚠️ Could not load admin cookies ({res.status_code})")
            return None
        raw = (res.json() or {}).get("cookiesTxt") or ""
        if not str(raw).strip():
            return None
        path = write_cookie_file(str(raw), Path("/tmp/admin-youtube-cookies.txt"))
        log(f"🍪 Loaded admin cookies ({len(raw.splitlines())} lines)")
        return path
    except Exception as err:
        log(f"⚠️ Could not load admin cookies: {err}")
        return None


def write_cookies_from_secret() -> str | None:
    raw = get_secret("YOUTUBE_COOKIES", "COOKIES_TXT")
    if not raw:
        return None
    return write_cookie_file(raw, Path("/tmp/youtube-cookies.txt"))


def find_cookies(job: dict) -> str | None:
    explicit = (job.get("cookieFilePath") or "").strip()
    candidates: list[str] = []
    admin_path = (job.get("_adminCookieFile") or "").strip()
    if admin_path:
        candidates.append(admin_path)
    if explicit:
        candidates.append(explicit)
    secret_path = write_cookies_from_secret()
    if secret_path:
        candidates.append(secret_path)
    candidates.extend(COOKIE_CANDIDATES)
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_file() and path.name.lower() in COOKIE_FILENAMES:
                candidates.append(str(path))
    seen: set[str] = set()
    for path in candidates:
        if not path or path in seen:
            continue
        seen.add(path)
        if Path(path).is_file():
            return path
    return None


def prefer_mp4(path: str) -> str:
    base, _ext = os.path.splitext(path)
    if os.path.exists(base + ".mp4"):
        return base + ".mp4"
    return path


def ydl_cookie_opts(job: dict) -> dict:
    cookie_file = find_cookies(job)
    if cookie_file:
        log(f"🍪 Using cookies: {cookie_file}")
        return {"cookiefile": cookie_file}
    log(
        "⚠️ No YouTube cookies found. Upload cookies in admin, attach a Kaggle dataset "
        "(KAGGLE_COOKIE_DATASET), or add secret YOUTUBE_COOKIES — otherwise downloads often 403."
    )
    return {}


def is_youtube_forbidden(err: BaseException) -> bool:
    text = str(err).lower()
    return "403" in text or "forbidden" in text or "not a bot" in text


def ydl_youtube_base_opts(job: dict) -> dict:
    opts: dict = {
        "noplaylist": True,
        "retries": 10,
        "fragment_retries": 10,
        "extractor_retries": 3,
        "continuedl": True,
        "concurrent_fragment_downloads": 1,
        "remote_components": ["ejs:github"],
        **ydl_cookie_opts(job),
    }
    deno = deno_path()
    if deno:
        opts["js_runtimes"] = {"deno": {"path": deno}}
    return opts


def list_video_urls(job: dict) -> list[str]:
    from yt_dlp import YoutubeDL

    youtube_url = (job.get("youtubeUrl") or "").strip()
    if not youtube_url:
        raise ValueError("Missing youtubeUrl")
    if not job.get("isPlaylist"):
        return [youtube_url]

    opts: dict = {
        "quiet": True,
        "no_warnings": True,
        "extract_flat": True,
        **ydl_youtube_base_opts(job),
        "noplaylist": False,
    }
    playlist_range = (job.get("playlistRange") or "").strip()
    if playlist_range:
        opts["playlist_items"] = playlist_range

    with YoutubeDL(opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)

    if not info or info.get("_type") != "playlist":
        return [youtube_url]

    urls: list[str] = []
    for entry in info.get("entries") or []:
        if not entry:
            continue
        url = entry.get("webpage_url") or entry.get("url") or entry.get("id")
        if not url:
            continue
        if not str(url).startswith("http"):
            url = f"https://www.youtube.com/watch?v={url}"
        urls.append(str(url))
    if not urls:
        raise RuntimeError("Playlist had no video entries")
    log(f"📃 Playlist has {len(urls)} video(s); downloading one at a time")
    return urls


def download_youtube(job: dict, work_dir: Path) -> str:
    from tqdm import tqdm
    from yt_dlp import YoutubeDL

    youtube_url = (job.get("youtubeUrl") or "").strip()
    quality = job.get("quality") or "1080p"
    if quality not in QUALITY_FORMATS:
        raise ValueError(f"Unknown quality: {quality}")

    work_dir.mkdir(parents=True, exist_ok=True)
    state: dict = {"bar": None}

    def progress_hook(d):
        if d["status"] == "downloading":
            total = d.get("total_bytes") or d.get("total_bytes_estimate")
            downloaded = d.get("downloaded_bytes", 0)
            if state["bar"] is None and total:
                state["bar"] = tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc="⬇️ Downloading",
                )
            if state["bar"] is not None:
                state["bar"].n = downloaded
                state["bar"].refresh()
        elif d["status"] == "finished" and state["bar"] is not None:
            state["bar"].close()
            state["bar"] = None
            log("🔄 Merging audio/video (if needed)...")

    base_opts = {
        "merge_output_format": "mp4",
        "outtmpl": str(work_dir / "%(title)s.%(ext)s"),
        "progress_hooks": [progress_hook],
        "quiet": True,
        "no_warnings": True,
        **ydl_youtube_base_opts(job),
    }

    disk_status(work_dir)
    last_error: Exception | None = None
    path = ""
    for index, attempt in enumerate(YOUTUBE_DOWNLOAD_ATTEMPTS, start=1):
        fmt = (
            "best[ext=mp4]/best"
            if attempt["format_key"] == "combined"
            else QUALITY_FORMATS[quality]
        )
        log(f"⬇️ Downloading {youtube_url} at {quality} ({attempt['label']})")
        ydl_opts = {
            **base_opts,
            "format": fmt,
            "extractor_args": attempt["extractor_args"],
        }
        try:
            with YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(youtube_url, download=True)
                path = prefer_mp4(ydl.prepare_filename(info))
            last_error = None
            break
        except Exception as err:
            last_error = err
            if state["bar"] is not None:
                state["bar"].close()
                state["bar"] = None
            if index < len(YOUTUBE_DOWNLOAD_ATTEMPTS) and is_youtube_forbidden(err):
                log(f"⚠️ {attempt['label']} failed ({err}). Trying next YouTube client...")
                continue
            raise

    if last_error:
        raise last_error

    custom_title = (job.get("customTitle") or "").strip()
    if custom_title and Path(path).exists():
        new_path = work_dir / f"{custom_title}.mp4"
        if new_path.resolve() != Path(path).resolve():
            Path(path).replace(new_path)
            path = str(new_path)

    if not Path(path).exists():
        raise RuntimeError("Download finished but the video file is missing")

    size_mb = Path(path).stat().st_size / (1024 * 1024)
    log(f"✅ Downloaded {Path(path).name} ({size_mb:.1f} MB)")
    disk_status(work_dir)
    return path


def get_duration(filepath: str) -> float:
    cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", filepath]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    return float(json.loads(result.stdout)["format"]["duration"])


def extract_chunk(input_path: str, output_path: str, start: float, length: float) -> None:
    cmd = [
        "ffmpeg",
        "-y",
        "-ss",
        f"{start:.3f}",
        "-t",
        f"{length:.3f}",
        "-i",
        input_path,
        "-c",
        "copy",
        "-map",
        "0",
        "-avoid_negative_ts",
        "make_zero",
        output_path,
    ]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg split failed: {result.stderr[-500:]}")


def is_quota_like_text(text: str) -> bool:
    lower = text.lower()
    return any(hint in lower for hint in DAILY_LIMIT_HINTS)


def is_access_denied_text(text: str) -> bool:
    upper = text.upper()
    return "ACCESS_DENIED" in upper or "ACCESS_FORBIDDEN" in upper


def is_daily_limit_error(response) -> bool:
    return is_quota_like_text(response.text or "")


def extract_access_token(payload: object, raw: str) -> str:
    if not isinstance(payload, dict):
        raise RuntimeError(f"Dailymotion auth returned a non-object body: {raw[:500]}")
    token = payload.get("access_token") or payload.get("accessToken")
    if token:
        return str(token)
    raise RuntimeError(
        "Dailymotion auth returned no access_token. Password grant is gone — "
        "create a Private API key in Studio → Organization → API keys. "
        f"Response keys: {sorted(payload.keys())}. Body: {raw[:800]}"
    )


def dailymotion_login(credentials: dict) -> dict:
    import requests

    log("🔐 Authenticating with Dailymotion API v2 (client_credentials)...")
    response = requests.post(
        "https://oauth2.dailymotion.com/v2/token",
        data={
            "grant_type": "client_credentials",
            "client_id": credentials["client_id"],
            "client_secret": credentials["client_secret"],
            "scope": "video.manage account.read",
        },
        timeout=30,
    )
    raw = response.text
    try:
        payload = response.json()
    except ValueError as err:
        raise RuntimeError(
            f"Dailymotion auth returned non-JSON ({response.status_code}): {raw[:500]}"
        ) from err
    if response.status_code != 200:
        raise RuntimeError(f"Dailymotion auth failed: {response.status_code} {raw[:800]}")
    token = extract_access_token(payload, raw)
    profile_id = resolve_profile_id(token, credentials.get("profile_id"))
    log(f"✅ Logged in to Dailymotion (profile {profile_id})")
    return {"token": token, "profile_id": profile_id}


def list_accessible_profiles(token: str) -> list[dict[str, str]]:
    import requests

    res = requests.get(
        "https://api.dailymotion.com/v2/me",
        headers={"Authorization": f"Bearer {token}"},
        timeout=30,
    )
    if not res.ok:
        raise RuntimeError(
            "Could not load Dailymotion profiles from /v2/me. "
            f"{res.status_code} {res.text[:800]}"
        )
    payload = res.json()
    data = payload if isinstance(payload, dict) else {}
    raw = data.get("profiles") or []
    if isinstance(raw, dict):
        raw = raw.get("items") or raw.get("data") or []
    profiles: list[dict[str, str]] = []
    for profile in raw:
        if not isinstance(profile, dict):
            continue
        profile_id = profile.get("profile_id") or profile.get("id")
        if not profile_id:
            continue
        profiles.append(
            {
                "id": str(profile_id),
                "name": str(profile.get("name") or profile.get("display_name") or profile_id),
            }
        )
    return profiles


def resolve_profile_id(token: str, explicit: str | None) -> str:
    profiles = list_accessible_profiles(token)
    summary = ", ".join(f"{item['id']} ({item['name']})" for item in profiles) or "(none)"
    log(f"📺 Dailymotion profiles this API key can manage: {summary}")

    if explicit:
        if any(item["id"] == explicit for item in profiles):
            return explicit
        log(
            f"⚠️ DAILYMOTION_PROFILE_ID={explicit} is not in that list. "
            "It is often a public user/channel id, or the Private API key is limited to other channels."
        )
        if profiles:
            chosen = profiles[0]["id"]
            log(f"➡️ Using accessible profile {chosen} instead")
            return chosen
        raise RuntimeError(
            f"DAILYMOTION_PROFILE_ID={explicit} is not allowed for this Private API key. "
            "In Studio → Organization → API keys, include that channel on the key, "
            "or set the secret to a profile_id from GET /v2/me. "
            f"Accessible: {summary}"
        )

    if not profiles:
        raise RuntimeError(
            "This API key has no manageable Dailymotion profiles. "
            "In Studio → Organization → API keys, allow the channel you want to publish to, "
            "or set DAILYMOTION_PROFILE_ID to that channel's profile_id from GET /v2/me."
        )
    return profiles[0]["id"]


def upload_to_dailymotion(
    file_path: str,
    title: str,
    part_number: int | None,
    session: dict,
    channel: str,
    tags: str,
) -> dict:
    import requests
    from requests_toolbelt.multipart.encoder import MultipartEncoder, MultipartEncoderMonitor
    from tqdm import tqdm

    token = session["token"]
    profile_id = session["profile_id"]
    headers = {"Authorization": f"Bearer {token}"}
    res = requests.post(
        "https://api.dailymotion.com/v2/files/upload_sessions",
        headers=headers,
        timeout=30,
    )
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    res.raise_for_status()
    upload_payload = res.json()
    upload_url = upload_payload.get("upload_url") or upload_payload.get("url")
    if not upload_url:
        raise RuntimeError(f"Dailymotion upload session had no URL: {res.text[:800]}")

    file_size = Path(file_path).stat().st_size
    pbar = tqdm(
        total=file_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc=f"⬆️ {Path(file_path).name}",
    )
    start_time = time.time()

    def callback(monitor):
        try:
            pbar.n = monitor.bytes_read
            elapsed = time.time() - start_time
            if elapsed > 0 and monitor.bytes_read > 0:
                speed = monitor.bytes_read / elapsed
                remaining = max(file_size - monitor.bytes_read, 0)
                eta_sec = remaining / speed if speed > 0 else 0
                mins, secs = divmod(int(eta_sec), 60)
                pbar.set_postfix({"speed": f"{speed / 1024 / 1024:.2f} MB/s", "ETA": f"{mins}m {secs}s"})
            pbar.refresh()
        except Exception:
            pass

    with open(file_path, "rb") as handle:
        encoder = MultipartEncoder(fields={"file": (os.path.basename(file_path), handle, "video/mp4")})
        monitor = MultipartEncoderMonitor(encoder, callback)
        res = requests.post(
            upload_url,
            data=monitor,
            headers={"Content-Type": monitor.content_type},
            timeout=None,
        )
        if is_daily_limit_error(res):
            pbar.close()
            raise DailyLimitReached(res.text)
        res.raise_for_status()
        uploaded = res.json()
        video_url = uploaded.get("url") or uploaded.get("file_url")
        if not video_url:
            raise RuntimeError(f"Dailymotion file upload returned no url: {res.text[:800]}")

    pbar.n = file_size
    pbar.refresh()
    pbar.close()

    final_title = f"{title} - Part {part_number}" if part_number else title
    final_title = final_title[:255]
    tag_list = [item.strip() for item in (tags or "").split(",") if item.strip()]
    log(f"   📝 Publishing '{final_title}'...")
    res = requests.post(
        f"https://api.dailymotion.com/v2/profiles/{profile_id}/videos",
        headers={**headers, "Content-Type": "application/json", "Accept": "application/json"},
        json={
            "title": final_title,
            "category": channel,
            "visibility": "public",
            "is_for_kids": False,
            "tags": tag_list,
            "source": {"file_url": video_url},
        },
        timeout=60,
    )
    if is_daily_limit_error(res):
        raise DailyLimitReached(res.text)
    if res.status_code not in (200, 201):
        body = res.text[:800]
        if res.status_code in (401, 403) or is_access_denied_text(body):
            raise RuntimeError(
                f"Dailymotion denied publishing to profile {profile_id}. "
                "If earlier parts of this job already uploaded, this is often the daily "
                "cap (15 videos or 10 hours / 24h) rather than a bad API key. "
                "Otherwise check DAILYMOTION_PROFILE_ID against GET /v2/me. "
                f"{body}"
            )
        raise RuntimeError(f"Dailymotion publish failed ({res.status_code}): {body}")
    created = res.json()
    video_id = created.get("video_id") or created.get("id")
    if not video_id:
        raise RuntimeError(f"Dailymotion publish returned no video id: {res.text[:800]}")
    public_url = f"https://www.dailymotion.com/video/{video_id}"
    log(f"   ✅ {public_url}")
    return {"title": final_title, "videoId": video_id, "url": public_url}


def pause_message(reason: str, checkpoint: JobCheckpoint) -> str:
    return (
        f"{reason} Saved {len(checkpoint.urls)} URL(s). "
        f"Resume at part {checkpoint.next_part} ({format_hms(checkpoint.resume_at_seconds)})."
    )


def raise_pause(reason: str, checkpoint: JobCheckpoint, cause: BaseException | None = None) -> None:
    message = pause_message(reason, checkpoint)
    if cause is None:
        raise PipelineStop(message, checkpoint, paused=True)
    raise PipelineStop(message, checkpoint, paused=True) from cause


def split_upload_cleanup(
    video_path: str,
    session: dict,
    job: dict,
    checkpoint: JobCheckpoint,
    quota: SessionQuota,
    on_progress=None,
) -> None:
    chunk_seconds = int(job.get("splitHours") or 1) * 3600 + int(job.get("splitMinutes") or 55) * 60
    if chunk_seconds > DM_MAX_FILE_SECONDS:
        raise_pause(
            "Split length exceeds Dailymotion's 2-hour per-file limit. Use 1h 55m or less.",
            checkpoint,
        )
    channel = job.get("dailymotionChannel") or "news"
    tags = job.get("dailymotionTags") or "youtube,upload"
    title = (job.get("customTitle") or "").strip() or Path(video_path).stem
    duration = get_duration(video_path)
    start = checkpoint.resume_at_seconds
    if start <= 0:
        start = max(checkpoint.next_part - 1, 0) * float(chunk_seconds)
    part = checkpoint.next_part

    def upload_part(file_path: str, part_number: int, length: float, next_start: float) -> None:
        size = Path(file_path).stat().st_size
        if size > DM_MAX_FILE_BYTES:
            raise_pause(
                f"Part {part_number} is {size / (1024 ** 3):.2f} GB, over Dailymotion's 4 GB per-file limit. "
                "Lower quality or shorten the split.",
                checkpoint,
            )
        try:
            uploaded = upload_to_dailymotion(file_path, title, part_number, session, channel, tags)
        except DailyLimitReached as err:
            raise_pause(
                "Dailymotion standard daily limit reached (15 videos or 10 hours / 24h).",
                checkpoint,
                err,
            )
        except Exception as err:
            if checkpoint.urls and is_access_denied_text(str(err)):
                raise_pause(
                    "Dailymotion blocked the next publish after earlier parts succeeded. "
                    "This usually means the standard daily cap (15 videos or 10 hours / 24h).",
                    checkpoint,
                    err,
                )
            raise PipelineStop(str(err), checkpoint, paused=False) from err
        checkpoint.urls.append(uploaded)
        checkpoint.next_part = part_number + 1
        checkpoint.resume_at_seconds = next_start
        quota.record(length)
        if on_progress:
            on_progress(checkpoint)

    if duration <= chunk_seconds:
        if start > 0.5 or part > 1:
            log("ℹ️ Video is a single part and a checkpoint already exists — skipping re-upload")
            return
        reason = quota.would_exceed(duration)
        if reason:
            raise_pause(reason, checkpoint)
        log("ℹ️ Video is within the length limit — uploading without splitting")
        upload_part(video_path, part, duration, duration)
        return

    if start >= duration - 0.5:
        log(
            f"ℹ️ Resume point {format_hms(start)} is past the end of the video "
            f"({format_hms(duration)}) — nothing left to upload"
        )
        return

    if part > 1 or start > 0:
        log(f"▶️ Resuming at part {part} ({format_hms(start)})")
    while start < duration - 0.5:
        length = min(float(chunk_seconds), duration - start)
        reason = quota.would_exceed(length)
        if reason:
            raise_pause(reason, checkpoint)
        part_path = str(Path(video_path).with_name(f"_upload_part_{part:03d}.mp4"))
        log(f"✂️ Extracting part {part} ({format_hms(start)} → {format_hms(start + length)})")
        try:
            extract_chunk(video_path, part_path, start, length)
            upload_part(part_path, part, length, start + length)
        finally:
            remove_path(part_path)
            disk_status(WORK_DIR)
        start += length
        part += 1


def run_pipeline(job: dict, session: dict, on_progress=None) -> list[dict]:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = JobCheckpoint.from_job(job)
    quota = SessionQuota.from_job(job)
    if quota.videos >= DM_MAX_VIDEOS_PER_DAY or quota.seconds >= DM_MAX_SECONDS_PER_DAY:
        raise_pause(
            "Dailymotion standard daily limit is still in effect for this job "
            "(15 videos or 10 hours / 24h). Wait for the window to reset, then Resume.",
            checkpoint,
        )
    try:
        for index, video_url in enumerate(list_video_urls(job), start=1):
            log(f"\n===== Video {index}: {video_url} =====")
            single = {**job, "youtubeUrl": video_url, "isPlaylist": False}
            if job.get("isPlaylist"):
                single["customTitle"] = None
            downloaded = download_youtube(single, WORK_DIR)
            try:
                split_upload_cleanup(downloaded, session, single, checkpoint, quota, on_progress)
            finally:
                remove_path(downloaded)
                disk_status(WORK_DIR)
        return checkpoint.urls
    except PipelineStop:
        raise
    except Exception as err:
        raise PipelineStop(str(err), checkpoint, paused=False) from err
    finally:
        remove_path(WORK_DIR)


def load_dailymotion_credentials() -> dict:
    creds = {
        "client_id": get_secret("DAILYMOTION_CLIENT_ID"),
        "client_secret": get_secret("DAILYMOTION_CLIENT_SECRET"),
        "profile_id": get_secret("DAILYMOTION_PROFILE_ID") or None,
    }
    if not creds["client_id"] or not creds["client_secret"]:
        raise SystemExit(
            "Missing DAILYMOTION_CLIENT_ID / DAILYMOTION_CLIENT_SECRET. "
            "Use a Private API key from Dailymotion Studio → Organization → API keys."
        )
    return creds


def main() -> None:
    ensure_tools()
    api_base = get_secret("API_BASE")
    secret = get_secret("COLAB_JOB_SECRET", "JOB_SECRET")
    if not api_base or not secret:
        raise SystemExit("Set Kaggle secrets API_BASE and COLAB_JOB_SECRET")

    session = dailymotion_login(load_dailymotion_credentials())
    processed = 0
    while True:
        job = claim_job(api_base, secret)
        if job is None:
            break
        job_id = job.get("id")
        log(f"\n{'=' * 60}\n▶ Job {job_id} — {job.get('youtubeUrl')}\n{'=' * 60}")
        admin_cookies = fetch_admin_cookies(api_base, secret)
        if admin_cookies:
            job["_adminCookieFile"] = admin_cookies
        try:
            def on_progress(checkpoint: JobCheckpoint) -> None:
                if job_id is not None:
                    save_progress(api_base, secret, job_id, checkpoint)

            urls = run_pipeline(job, session, on_progress)
            if not urls:
                raise RuntimeError("Pipeline finished with no Dailymotion URLs")
            complete_job(api_base, secret, job_id, urls)
            log(f"✅ Reported {len(urls)} URL(s) for job #{job_id}")
            for item in urls:
                log(f"   • {item['title']}: {item['url']}")
        except PipelineStop as stop:
            kind = "paused" if stop.paused else "failed"
            log(f"{'⏸️' if stop.paused else '❌'} Job {job_id} {kind}: {stop}")
            if job_id is not None:
                fail_job(
                    api_base,
                    secret,
                    job_id,
                    str(stop),
                    paused=stop.paused,
                    checkpoint=stop.checkpoint,
                )
        except Exception as err:
            log(f"❌ Job {job_id} failed: {err}")
            if job_id is not None:
                fail_job(api_base, secret, job_id, str(err), checkpoint=JobCheckpoint.from_job(job))
        processed += 1
        remove_path(WORK_DIR)

    remove_path(WORK_DIR)
    if processed == 0:
        log("No pending jobs")
    else:
        log(f"Done. Processed {processed} job(s).")


if __name__ == "__main__":
    main()
